# Replay a bounded judge

Python 3.12 · offline · estimated time: 18 minutes

You will turn the Lesson 01 cases into strict, replayable judge predictions. The notebook supplies cached responses, validation, alignment, and persistence for immediate feedback.

Replace every `TODO`, then run each cell in order. The final cell persists `judge_predictions.jsonl`, which Lesson 03 will consume.


In [1]:
import json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'build/lesson-01/eval_cases.jsonl').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
cases_path = ROOT / 'build/lesson-01/eval_cases.jsonl'
cases = [json.loads(line) for line in cases_path.read_text(encoding='utf-8').splitlines()]
assert cases, 'eval_cases.jsonl has no cases'
assert len(cases) >= 10, 'Lesson 03 requires at least 10 cases'
assert {row['human_label'] for row in cases} <= {0, 1}
print(f'CHECK 1 — loaded {len(cases)} cases')
print('CHECK 1 — inherited mapping: human_label 1 = regression present; 0 = regression absent')


CHECK 1 — loaded 10 cases
CHECK 1 — inherited mapping: human_label 1 = regression present; 0 = regression absent


In [2]:
# TODO 1: Write the bounded rubric and map its two verdicts to binary labels.
# Use only input and candidate_output; never reveal human_label to the judge.
RUBRIC_VERSION = 'refund-window-v1'
RUBRIC = ('Decide only whether candidate_output grants a refund when input says the request is outside 30 days. '
          'Return exactly one verdict: REGRESSION_PRESENT or REGRESSION_ABSENT, plus a brief reason. '
          'Use only input and candidate_output; ignore tone, polish, and unrelated quality.')
VERDICTS = {'REGRESSION_PRESENT': 1, 'REGRESSION_ABSENT': 0}

assert RUBRIC.strip(), 'TODO 1: write the bounded refund-window rubric'
assert VERDICTS == {'REGRESSION_PRESENT': 1, 'REGRESSION_ABSENT': 0}, 'TODO 1: map both verdicts to 1 and 0'
example = cases[0]
request = {'rubric_version': RUBRIC_VERSION, 'rubric': RUBRIC, 'input': example['input'], 'candidate_output': example['candidate_output']}
assert 'human_label' not in request and 'human_label' not in json.dumps(request)
print('CHECK 2 — request fields:', sorted(request))
print('CHECK 2 — human_label excluded; verdict mapping fixed: PASS')


CHECK 2 — request fields: ['candidate_output', 'input', 'rubric', 'rubric_version']
CHECK 2 — human_label excluded; verdict mapping fixed: PASS


In [3]:
# TODO 2: Declare the exact fields allowed in one cached judge response.
FIXTURE_FIELDS = {'case_id', 'verdict', 'reason'}
assert FIXTURE_FIELDS == {'case_id', 'verdict', 'reason'}, 'TODO 2: declare the three-field response contract'
fixture_path = ROOT / 'build/lesson-02/cached_judge_responses.jsonl'
responses = [json.loads(line) for line in fixture_path.read_text(encoding='utf-8').splitlines()]
errors, seen = [], set()
for response in responses:
    case_id = response.get('case_id', '<missing case_id>')
    if set(response) != FIXTURE_FIELDS: errors.append(f'{case_id}: fields must be {sorted(FIXTURE_FIELDS)}')
    if not isinstance(response.get('case_id'), str) or not response.get('case_id', '').strip(): errors.append(f'{case_id}: case_id must be a non-empty string')
    if response.get('verdict') not in VERDICTS: errors.append(f'{case_id}: verdict must be one of {sorted(VERDICTS)}')
    if not isinstance(response.get('reason'), str) or not response.get('reason', '').strip(): errors.append(f'{case_id}: reason must be non-empty')
    if case_id in seen: errors.append(f'{case_id}: duplicate cached response')
    seen.add(case_id)
case_ids = {row['case_id'] for row in cases}
unknown, missing = seen - case_ids, case_ids - seen
if unknown: errors.append(f'unknown case_id(s): {sorted(unknown)}')
if missing: errors.append(f'missing response(s) for: {sorted(missing)}')
if errors: raise ValueError('\n'.join(errors))
print('CHECK 3 — allowed verdicts, non-empty reasons, unique IDs, exact coverage: PASS')

CHECK 3 — allowed verdicts, non-empty reasons, unique IDs, exact coverage: PASS


In [4]:
by_id = {response['case_id']: response for response in responses}
predictions = [
    {'case_id': case['case_id'], 'human_label': case['human_label'],
     'judge_label': VERDICTS[by_id[case['case_id']]['verdict']],
     'judge_reason': by_id[case['case_id']]['reason']}
    for case in cases
]
assert [(row['case_id'], row['human_label']) for row in predictions] == [(row['case_id'], row['human_label']) for row in cases]
print('CHECK 4 — aligned by case_id; case_id and human_label unchanged: PASS')

CHECK 4 — aligned by case_id; case_id and human_label unchanged: PASS


In [5]:
disagreements = [row for row in predictions if row['human_label'] != row['judge_label']]
print('CHECK 5 — disagreements (case_id | human_label | judge_label | judge_reason)')
for row in disagreements:
    print(f"{row['case_id']} | {row['human_label']} | {row['judge_label']} | {row['judge_reason']}")
artifact_path = ROOT / 'build/lesson-02/judge_predictions.jsonl'
artifact_path.write_text(''.join(json.dumps(row) + '\n' for row in predictions), encoding='utf-8')
reloaded = [json.loads(line) for line in artifact_path.read_text(encoding='utf-8').splitlines()]
assert reloaded == predictions, 'read-back predictions differ from in-memory records'
print(f'CHECK 5 — wrote and reloaded {len(reloaded)} predictions: {artifact_path.relative_to(ROOT)}')
print('FINAL PASS — artifact ready for EP-03')

CHECK 5 — disagreements (case_id | human_label | judge_label | judge_reason)
refund-window-003 | 1 | 0 | The reply discusses a three-day delay but does not explicitly say the customer is eligible.
refund-window-007 | 1 | 0 | The reply frames the late refund as an exception rather than explicitly calling it eligible.
refund-window-008 | 0 | 1 | The reply approves a refund, which this cached judge incorrectly treats as a regression.
CHECK 5 — wrote and reloaded 10 predictions: build\lesson-02\judge_predictions.jsonl
FINAL PASS — artifact ready for EP-03
